# COSMIC-LATENT FRONTIER  
## Speculative Cartography of Unresolved Cosmological Structure

Name: Sabrina Palis

Repository: https://github.com/MinervaRose/cosmic-latent-governance

### Notebook 2 — Frontier Visualization & Anomaly Dossier Lab

This notebook is the speculative companion to **Notebook 1: Methods & Governance**.

Notebook 1 established the rigorous pipeline:

- SDSS + Euclid + IllustrisTNG + Galaxy Zoo ingestion
- normalized representation-space geometry
- information geometry
- latent manifold learning
- anomaly detection
- governance and epistemic tagging

Notebook 2 does **not** redo the core method.  
It takes the processed outputs from Notebook 1 and asks:

> What can we see if we treat anomalies as frontier objects rather than merely outliers?

This notebook is intentionally more cinematic and speculative, but still bounded:

- no claim of new physics
- no claim of extraterrestrial intelligence
- no claim of dark matter discovery
- no claim that latent-space geometry is physical spacetime

Instead, the notebook creates visual instruments for exploring:

- unknownness
- epistemic risk
- anomaly neighbourhoods
- unresolved morphology
- dataset disagreement
- frontier candidates

## Epistemic Boundary

This notebook is an exploratory visualization annex.

The working concept is:

> **epistemic frontier object**: a data object or region that is unusual, uncertain, difficult to reconstruct, weakly classified, or located near a boundary between representation regimes.

These are not discoveries.  
They are prompts for closer inspection.

The goal is not to prove anything extraordinary.  
The goal is to build visual instruments for asking better questions.

# Module 0 — Setup

In [16]:
import os
import json
import zipfile
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

import networkx as nx

SEED = 42
np.random.seed(SEED)

print("COSMIC-LATENT FRONTIER initialized.")

COSMIC-LATENT FRONTIER initialized.


In [15]:
!pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.1 MB/s eta 0:00:00


# Module 1 — Load Notebook 1 Outputs

This notebook expects the processed export from Notebook 1.

It first checks whether `cosmic_df` already exists in memory.  
If not, it tries to load:

- `/content/cosmic_latent_exports/cosmic_latent_processed_multisource.csv`
- `/content/cosmic_latent_exports.zip`
- `/content/cosmic_latent_exports/cosmic_df.csv`

If needed, upload your `cosmic_latent_exports.zip` from Notebook 1 into Colab.

In [17]:
# ============================================================
# LOAD NOTEBOOK 1 EXPORTS (FLEXIBLE VERSION)
# ============================================================

import os
import zipfile
import pandas as pd

def find_csv(possible_names):

    search_dirs = [
        "/content",
        "/content/cosmic_latent_exports",
    ]

    for directory in search_dirs:

        if not os.path.exists(directory):
            continue

        files = os.listdir(directory)

        for f in files:

            f_lower = f.lower()

            for target in possible_names:

                if target.lower() in f_lower and f_lower.endswith(".csv"):

                    return os.path.join(directory, f)

    return None


# ------------------------------------------------------------
# EXTRACT ZIP IF PRESENT
# ------------------------------------------------------------

zip_candidates = [
    "/content/cosmic_latent_exports.zip",
]

for zip_path in zip_candidates:

    if os.path.exists(zip_path):

        extract_dir = "/content/cosmic_latent_exports"

        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as zipf:
            zipf.extractall(extract_dir)

        print(f"Extracted ZIP: {zip_path}")


# ------------------------------------------------------------
# FIND MAIN COSMIC DATAFRAME
# ------------------------------------------------------------

main_csv = find_csv([
    "cosmic_latent_processed_multisource",
    "cosmic_df",
])

if main_csv is None:

    raise FileNotFoundError(
        "Could not locate processed cosmic dataframe CSV."
    )

print("Loading:", main_csv)

cosmic_df = pd.read_csv(main_csv)

print("cosmic_df shape:", cosmic_df.shape)

display(cosmic_df.head())


# ------------------------------------------------------------
# OPTIONAL SUPPORT FILES
# ------------------------------------------------------------

optional_files = {
    "governance_summary": [
        "governance_summary",
        "cosmic_latent_governance_summary",
    ],

    "galaxy_zoo": [
        "galaxy_zoo",
        "galaxy_zoo_morphology",
    ],

    "sdss_processed": [
        "sdss_processed",
    ],

    "euclid_processed": [
        "euclid_processed",
    ],

    "tng_processed": [
        "tng_processed",
    ],
}

loaded_support = {}

for key, patterns in optional_files.items():

    path = find_csv(patterns)

    if path:

        try:
            loaded_support[key] = pd.read_csv(path)
            print(f"Loaded support file: {path}")

        except Exception as e:
            print(f"Failed loading {path}: {repr(e)}")

print("\nLoaded support datasets:")
print(list(loaded_support.keys()))

Extracted ZIP: /content/cosmic_latent_exports.zip
Loading: /content/cosmic_latent_exports/cosmic_latent_processed_multisource.csv
cosmic_df shape: (4250, 89)


,objid,ra,dec,redshift,u,g,r,i,zmag,x,...,iso_anomaly_flag,graph_anomaly_score,iso_anomaly_score_norm,graph_anomaly_score_norm,ae_reconstruction_error_norm,cosmic_anomaly_index,cosmic_anomaly_flag,evidence_score,interpretive_risk,epistemic_tag
0,1.237649e+18,196.554898,-0.664147,0.083987,19.00659,17.46996,16.69207,16.29661,16.01036,-349.590818,...,0,1.101445,0.172683,0.179263,0.000666,0.123052,0,0.7,0.196955,observed
1,1.237649e+18,196.591389,-0.827959,0.190357,20.55509,18.50240,17.25141,16.77685,16.43557,-771.305055,...,0,1.781245,0.125925,0.290543,0.027963,0.145922,0,0.7,0.236677,observed
2,1.237649e+18,196.612641,-0.806041,0.084177,18.73435,16.83528,15.89845,15.49562,15.14348,-350.248702,...,0,2.537666,0.235155,0.414366,0.006526,0.220329,0,0.7,0.365916,observed
3,1.237649e+18,196.623459,-0.797385,0.084211,19.27567,17.23888,16.27929,15.86932,15.52172,-350.367110,...,0,2.358018,0.206299,0.384958,0.005510,0.199660,0,0.7,0.330015,observed
4,1.237649e+18,214.120630,-0.743875,0.048742,17.08624,15.41548,14.67285,14.25464,13.94828,-176.724747,...,0,0.940335,0.077162,0.152890,0.002341,0.077434,0,0.7,0.117721,observed


Loaded support file: /content/cosmic_latent_exports/cosmic_latent_governance_summary.csv
Loaded support file: /content/cosmic_latent_exports/galaxy_zoo_morphology.csv
Loaded support file: /content/cosmic_latent_exports/sdss_processed.csv
Loaded support file: /content/cosmic_latent_exports/euclid_processed.csv
Loaded support file: /content/cosmic_latent_exports/tng_processed.csv

Loaded support datasets:
['governance_summary', 'galaxy_zoo', 'sdss_processed', 'euclid_processed', 'tng_processed']


## Required Columns Check

Notebook 2 needs the core outputs created by Notebook 1.

If a required column is missing, this cell tells you exactly what is absent.

In [18]:
required_cols = [
    "dataset_name",
    "umap_1",
    "umap_2",
    "cosmic_anomaly_index",
    "evidence_score",
    "interpretive_risk",
    "epistemic_tag",
]

missing = [c for c in required_cols if c not in cosmic_df.columns]

if missing:
    print("Missing required columns:", missing)
    print("Rerun Notebook 1 through anomaly detection + governance, then export again.")
else:
    print("All required Notebook 1 columns are present.")

All required Notebook 1 columns are present.


# Module 2 — Frontier Metrics

Notebook 1 produced:
- anomaly score
- evidence score
- interpretive risk
- epistemic tags

Notebook 2 constructs additional speculative visualization metrics:

## Unknownness Index

A composite measure of unresolvedness:

- high anomaly
- high interpretive risk
- low evidence support

## Frontier Tension

A measure of conflict between anomaly and evidence:

- high anomaly + high evidence = intriguing but inspectable
- high anomaly + low evidence = fragile/speculative
- low anomaly + high evidence = stable

In [19]:
# ============================================================
# FRONTIER METRICS
# ============================================================

for col in ["cosmic_anomaly_index", "interpretive_risk", "evidence_score"]:
    cosmic_df[col] = pd.to_numeric(cosmic_df[col], errors="coerce").fillna(0)

cosmic_df["unknownness_index"] = (
    0.45 * cosmic_df["cosmic_anomaly_index"]
    + 0.35 * cosmic_df["interpretive_risk"]
    + 0.20 * (1 - cosmic_df["evidence_score"])
)

cosmic_df["frontier_tension"] = (
    cosmic_df["cosmic_anomaly_index"] * (1 - cosmic_df["evidence_score"])
)

# Normalize for visual comparability
for col in ["unknownness_index", "frontier_tension"]:
    cosmic_df[col + "_norm"] = MinMaxScaler().fit_transform(cosmic_df[[col]])

display(
    cosmic_df[
        [
            "dataset_name",
            "cosmic_anomaly_index",
            "interpretive_risk",
            "evidence_score",
            "unknownness_index",
            "frontier_tension"
        ]
    ].head()
)

,dataset_name,cosmic_anomaly_index,interpretive_risk,evidence_score,unknownness_index,frontier_tension
0,SDSS,0.123052,0.196955,0.7,0.184308,0.036916
1,SDSS,0.145922,0.236677,0.7,0.208502,0.043776
2,SDSS,0.220329,0.365916,0.7,0.287219,0.066099
3,SDSS,0.199660,0.330015,0.7,0.265352,0.059898
4,SDSS,0.077434,0.117721,0.7,0.136048,0.023230


## Interpretation — Unknownness

The unknownness index does not mean:

- extraterrestrial intelligence,
- impossible physics,
- dark matter detection,
- or evidence of new cosmological laws.

Instead, it measures something narrower and more computational:

> how difficult an object is for the current representation system to explain coherently.

Unknownness increases when an object becomes:

- structurally anomalous,
- weakly supported by evidence,
- unstable inside latent space,
- difficult to cluster with neighbouring objects,
- or inconsistent across observational descriptors.

In other words:

the system is not saying
“this object is extraordinary.”

It is saying:

> “this object resists compression into the current learned representation.”

This notebook therefore treats unknownness as an exploratory computational property rather than a scientific claim.

# Module 3 — Epistemic Frontier Map

This is the central visualization of Notebook 2.

Instead of treating latent space as a simple clustering diagram, this module treats it as a navigable epistemic terrain:

- regions of stability,
- regions of ambiguity,
- regions of structural tension,
- and regions where the representation system begins to fail.

Each point in the manifold corresponds to a cosmological object projected into learned representation space.

Distances inside this space do not necessarily represent physical distance.

Instead, they represent similarity of structure, morphology, statistical behaviour, and observational relationships.

The frontier map therefore becomes a kind of cartography of unresolved representation.

Some regions appear dense and coherent.
Others fragment into isolated filaments, detached islands, sparse bridges, or unstable neighbourhoods.

These regions should not be interpreted as discoveries.

They are computational zones where:

- the model compresses information poorly,
- anomaly scores increase,
- topology becomes unstable,
- or classification confidence weakens.

In this sense, the frontier map acts less like a telescope and more like an instrument for locating epistemic stress inside high-dimensional cosmological data.

In [20]:
fig_epistemic_frontier = px.scatter(
    cosmic_df,
    x="umap_1",
    y="umap_2",
    color="unknownness_index",
    symbol="dataset_name",
    opacity=0.82,
    hover_data=[
        "dataset_name",
        "epistemic_tag",
        "cosmic_anomaly_index",
        "evidence_score",
        "interpretive_risk",
        "unknownness_index",
    ],
    title="Epistemic Frontier Map — Machine Cartography of Unresolved Structure",
)

fig_epistemic_frontier.update_layout(height=760)
fig_epistemic_frontier.show()

## Interpretation — Epistemic Frontier Map

This is not a map of physical space.

It is a map of how the representation system organizes cosmological objects after compressing many observational properties into a shared latent structure.

Nearby points are not necessarily physically close in the universe.

Instead, they are structurally similar according to the learned representation.

Bright regions indicate areas where the system detects combinations of:

- elevated anomaly,
- interpretive instability,
- weak evidence support,
- or unresolved structural tension.

The most interesting regions are not isolated “weird points” alone.

They are:

- borders,
- detached islands,
- thin bridges,
- fragmented filaments,
- and transition zones where the learned organization becomes unstable.

These regions can be interpreted as zones of epistemic stress:

areas where the current representation system struggles to compress the observational structure into a coherent internal geometry.

The frontier map therefore acts as a form of machine-assisted cartography of unresolved cosmological organization under uncertainty.

# Module 4 — Frontier Quadrant Map

This visualization separates anomaly from evidence quality.

Its purpose is simple but important:

> not all strange objects are equally meaningful.

Some objects appear unusual because they contain genuinely rare structural combinations.

Others appear unusual because the observational support is weak, noisy, incomplete, blended, or unstable.

The quadrant map therefore asks a more disciplined question:

- Is this object structurally difficult to explain?
- Or is the system simply uncertain because the evidence quality is poor?

This distinction is essential in exploratory AI-assisted cosmology.

Without it, anomaly detection easily collapses into overinterpretation.

The map can therefore be read as a form of epistemic filtering:

a way to separate:

- robust structural tension,
- weak observational support,
- transitional representation states,
- and potentially meaningful unresolved organization.

In this sense, the quadrant map acts as a governance instrument for speculative interpretation.

It does not eliminate curiosity.

It constrains curiosity within explicit uncertainty boundaries.

In [21]:
fig_frontier_quadrants = px.scatter(
    cosmic_df,
    x="evidence_score",
    y="cosmic_anomaly_index",
    color="epistemic_tag",
    symbol="dataset_name",
    size="unknownness_index_norm",
    hover_data=[
        "dataset_name",
        "unknownness_index",
        "interpretive_risk",
        "source_label" if "source_label" in cosmic_df.columns else "dataset_name",
    ],
    title="Frontier Quadrant Map — Anomaly vs Evidence",
)

fig_frontier_quadrants.add_hline(
    y=cosmic_df["cosmic_anomaly_index"].quantile(0.94),
    line_dash="dash",
    annotation_text="Top anomaly threshold"
)

fig_frontier_quadrants.add_vline(
    x=0.5,
    line_dash="dot",
    annotation_text="Evidence midpoint"
)

fig_frontier_quadrants.update_layout(height=700)
fig_frontier_quadrants.show()

## Interpretation — Frontier Quadrants

This figure compares two quantities:

- anomaly intensity,
- and evidence quality.

The horizontal axis represents how reliable or observationally supported an object appears within the current pipeline.

The vertical axis represents how structurally unusual the object appears according to the anomaly system developed in Notebook 1.

The dashed threshold lines divide the space into different interpretive regimes.

Most objects occupy the lower-right region:

- relatively strong evidence,
- and relatively low anomaly.

These form the stable background population of the representation space.

Objects in the upper regions exhibit elevated anomaly scores.

However, the interpretation differs depending on evidence quality:

- High anomaly + high evidence may indicate structurally unusual but well-supported candidates.
- High anomaly + low evidence may instead reflect noisy, blended, incomplete, or uncertain observations.

The purpose of this map is therefore not to identify “mysterious objects.”

Its purpose is to constrain interpretation.

The figure acts as a governance layer that separates:

- statistically unusual structure,
- from uncertainty introduced by weak observational support.

This helps prevent anomaly detection from collapsing into unrestricted speculation.

# Module 5 — Frontier Sectors

The previous modules visualized individual objects and local anomaly structure.

This module shifts from isolated points to regional organization.

The latent manifold is partitioned into computational sectors using clustering methods applied directly in representation space.

These sectors do not correspond to physical regions of the universe.

Instead, they represent regions of shared structural similarity inside the learned latent geometry.

For each sector, the notebook summarizes quantities such as:

- average anomaly intensity,
- evidence quality,
- unknownness,
- frontier tension,
- and governance composition.

This transforms the latent manifold into a navigable frontier atlas:

a coarse-grained map of where unresolved structure accumulates inside the representation system.

Some sectors appear stable and internally coherent.

Others concentrate elevated anomaly scores, fragmented topology, or weak interpretive stability.

The goal is not to claim discovery.

It is to identify regions where further investigation may be computationally interesting under explicit uncertainty constraints.

In [22]:
# ============================================================
# FRONTIER SECTORS
# ============================================================

sector_df = cosmic_df.dropna(subset=["umap_1", "umap_2"]).copy()

N_SECTORS = 8

kmeans = KMeans(n_clusters=N_SECTORS, random_state=SEED, n_init=10)
sector_df["frontier_sector"] = kmeans.fit_predict(sector_df[["umap_1", "umap_2"]])

cosmic_df = cosmic_df.merge(
    sector_df[["frontier_sector"]],
    left_index=True,
    right_index=True,
    how="left"
)

sector_summary = (
    cosmic_df
    .groupby("frontier_sector")
    .agg(
        n=("unknownness_index", "size"),
        mean_unknownness=("unknownness_index", "mean"),
        max_unknownness=("unknownness_index", "max"),
        mean_anomaly=("cosmic_anomaly_index", "mean"),
        mean_evidence=("evidence_score", "mean"),
    )
    .reset_index()
    .sort_values("mean_unknownness", ascending=False)
)

display(sector_summary)

fig_frontier_sectors = px.scatter(
    cosmic_df,
    x="umap_1",
    y="umap_2",
    color="frontier_sector",
    size="unknownness_index_norm",
    symbol="dataset_name",
    hover_data=[
        "dataset_name",
        "frontier_sector",
        "unknownness_index",
        "cosmic_anomaly_index",
        "evidence_score"
    ],
    title="Frontier Sectors — Latent Terrain Partitioned into Exploratory Zones",
)

fig_frontier_sectors.update_layout(height=760)
fig_frontier_sectors.show()

,frontier_sector,n,mean_unknownness,max_unknownness,mean_anomaly,mean_evidence
4,4,486,0.302364,0.717452,0.223811,0.642691
5,5,384,0.262241,0.630176,0.186747,0.647256
2,2,441,0.223980,0.447807,0.145700,0.621436
1,1,518,0.223931,0.509815,0.147543,0.631429
6,6,163,0.222075,0.499862,0.145853,0.631773
7,7,814,0.190189,0.487349,0.128194,0.697789
0,0,788,0.181476,0.470124,0.118493,0.690038
3,3,656,0.159748,0.447759,0.098825,0.694650


## Interpretation — Frontier Sectors

The frontier sector view transforms the latent manifold into a navigable atlas of representation-space regions.

Instead of asking:

> Which individual object appears unusual?

the notebook asks:

> Which regions of latent space accumulate unresolved structural tension?

Each colored sector represents a computational region identified through clustering in the learned representation geometry.

These sectors are not physical regions of the universe.

They are regions of shared structural similarity inside the latent manifold constructed from observational and derived features.

Some sectors appear:

- compact,
- internally coherent,
- and geometrically stable.

Others appear:

- fragmented,
- elongated,
- sparsely connected,
- or positioned near transition boundaries.

These differences may indicate regions where the representation system organizes information with different levels of confidence or structural consistency.

The purpose of this visualization is exploratory rather than confirmatory.

It treats the latent manifold as a terrain that can be surveyed, partitioned, and prioritized for further investigation under explicit uncertainty constraints.

In this sense, the notebook shifts from point-based anomaly detection toward machine-assisted cartography of unresolved representation structure.

# Module 6 — Anomaly Dossier System

This module transforms frontier candidates into structured investigative records.

Rather than treating anomalies as isolated points on a visualization, the notebook assigns stable dossier entries to the highest-ranked frontier objects.

Each candidate receives:

- a frontier identifier,
- dataset provenance,
- latent-space coordinates,
- anomaly metrics,
- evidence estimates,
- interpretive risk,
- governance status,
- and local representation context.

These dossiers are not discoveries.

They are computational case files generated by the exploratory pipeline.

Their purpose is not to claim new astrophysical phenomena.

Their purpose is to preserve:

- traceability,
- reproducibility,
- interpretive discipline,
- and systematic prioritization for future inspection.

In this framework, anomaly detection becomes less like “finding strange dots” and more like maintaining a governed archive of unresolved computational structure.

The notebook therefore shifts from visualization into exploratory frontier bookkeeping:

a structured registry of objects that resist easy representation within the current latent system.

In [23]:
# ============================================================
# ANOMALY DOSSIER SYSTEM
# ============================================================

DOSSIER_TOP_N = 25

frontier_dossier = (
    cosmic_df
    .sort_values("unknownness_index", ascending=False)
    .head(DOSSIER_TOP_N)
    .copy()
)

frontier_dossier["frontier_id"] = [
    f"CL-FRONTIER-{i+1:03d}"
    for i in range(len(frontier_dossier))
]

dossier_cols = [
    "frontier_id",
    "dataset_name",
    "source_label",
    "ra",
    "dec",
    "objid",
    "id",
    "x_norm",
    "y_norm",
    "z_norm",
    "umap_1",
    "umap_2",
    "ae_latent_1",
    "ae_latent_2",
    "frontier_sector",
    "unknownness_index",
    "frontier_tension",
    "cosmic_anomaly_index",
    "iso_anomaly_score",
    "graph_anomaly_score",
    "ae_reconstruction_error",
    "evidence_score",
    "interpretive_risk",
    "epistemic_tag",
]

dossier_cols = [c for c in dossier_cols if c in frontier_dossier.columns]

display(frontier_dossier[dossier_cols])

,frontier_id,dataset_name,source_label,ra,dec,objid,id,x_norm,y_norm,z_norm,...,frontier_sector,unknownness_index,frontier_tension,cosmic_anomaly_index,iso_anomaly_score,graph_anomaly_score,ae_reconstruction_error,evidence_score,interpretive_risk,epistemic_tag
2350,CL-FRONTIER-001,Euclid,Euclid_Q1_COSMOS_MER,269.742957,66.033976,NaN,NaN,0.298411,0.683506,-1.074231,...,4,0.717452,0.304477,0.585394,0.568260,0.136193,6.619893,0.479878,1.000000,speculative
3041,CL-FRONTIER-002,Euclid,Euclid_Q1_COSMOS_MER,269.721246,65.993703,NaN,NaN,-0.188348,-1.516282,0.258541,...,4,0.697295,0.294567,0.566341,0.619196,1.039587,3.903310,0.479878,0.966907,speculative
3487,CL-FRONTIER-003,Euclid,Euclid_Q1_COSMOS_MER,269.782354,66.043101,NaN,NaN,1.181740,1.181918,0.817248,...,4,0.666948,0.257208,0.547108,0.580006,0.112964,5.496632,0.529878,0.933500,speculative
3947,CL-FRONTIER-004,Euclid,Euclid_Q1_COSMOS_MER,269.814526,66.008484,NaN,NaN,1.903047,-0.708901,1.373021,...,4,0.638287,0.265555,0.510564,0.602928,0.518489,3.658248,0.479878,0.870026,speculative
2970,CL-FRONTIER-005,Euclid,Euclid_Q1_COSMOS_MER,269.731482,66.045618,NaN,NaN,0.041141,1.319365,0.153750,...,4,0.636860,0.243837,0.518667,0.591764,0.853470,3.762921,0.529878,0.884101,speculative
3698,CL-FRONTIER-006,Euclid,Euclid_Q1_COSMOS_MER,269.649459,66.011309,NaN,NaN,-1.797890,-0.554616,1.101243,...,5,0.630176,0.189636,0.534103,0.644655,1.053145,2.518863,0.644945,0.910911,inferred
3982,CL-FRONTIER-007,Euclid,Euclid_Q1_COSMOS_MER,269.815306,66.008314,NaN,NaN,1.920535,-0.718189,1.413782,...,5,0.625660,0.188607,0.529635,0.671422,0.351494,2.487970,0.643892,0.903151,inferred
3667,CL-FRONTIER-008,Euclid,Euclid_Q1_COSMOS_MER,269.657764,66.003749,NaN,NaN,-1.611678,-0.967571,1.065769,...,4,0.620426,0.256774,0.493680,0.589548,0.374561,3.787865,0.479878,0.840701,inferred
2741,CL-FRONTIER-009,Euclid,Euclid_Q1_COSMOS_MER,269.728771,65.998099,NaN,NaN,-0.019643,-1.276175,-0.273322,...,5,0.617142,0.255160,0.490577,0.560525,0.864905,3.939021,0.479878,0.835310,inferred
3613,CL-FRONTIER-010,Euclid,Euclid_Q1_COSMOS_MER,269.779885,66.045945,NaN,NaN,1.126384,1.337226,0.991832,...,4,0.616340,0.185087,0.520985,0.652194,1.501673,1.548595,0.644736,0.888126,inferred


## Interpretation — Frontier Dossiers

The frontier dossiers convert abstract anomaly structure into persistent investigative records.

Each row corresponds to a candidate object identified by the exploratory pipeline as occupying a region of elevated unresolved representation structure.

The dossier system preserves:

- provenance,
- traceability,
- ranking,
- governance state,
- and latent-space context.

This allows frontier candidates to be revisited, compared, filtered, and tracked across future notebook iterations.

Importantly, the dossier ranking is not based on a single anomaly score alone.

Each candidate emerges from the interaction between:

- statistical anomaly,
- graph irregularity,
- reconstruction instability,
- evidence quality,
- and interpretive governance constraints.

The resulting frontier registry therefore acts less like a catalogue of “discoveries” and more like a structured archive of unresolved computational cases.

Several visible patterns emerge:

- many high-ranking candidates cluster inside specific frontier sectors,
- Euclid candidates dominate the upper frontier regime,
- speculative tags appear primarily where anomaly and interpretive risk remain simultaneously elevated,
- while inferred candidates often occupy intermediate uncertainty states.

These dossiers should not be interpreted as evidence of unknown astrophysical phenomena.

They represent objects that the current representation system compresses imperfectly under the chosen observational and geometric framework.

# Module 7 — Labelled Frontier Map

The previous module generated structured frontier dossiers.

This module places those candidates back into the latent terrain from which they emerged.

Each labelled point corresponds to a persistent frontier case file projected into representation space.

The purpose is not only to inspect isolated objects, but to examine their positional context inside the broader latent geometry.

This allows the notebook to ask questions such as:

- Do frontier candidates cluster together?
- Do they emerge near sector boundaries?
- Do speculative objects accumulate inside unstable regions?
- Are some candidates isolated while others belong to coherent anomalous neighbourhoods?

The labelled frontier map therefore acts as a navigational layer for exploratory analysis.

Instead of treating anomalies as disconnected numerical scores, the notebook repositions them inside the global structure of the learned manifold.

In this sense, frontier candidates become cartographic landmarks within the broader epistemic terrain.

In [24]:
fig_frontier_dossier_map = px.scatter(
    cosmic_df,
    x="umap_1",
    y="umap_2",
    color="unknownness_index",
    symbol="dataset_name",
    opacity=0.30,
    hover_data=[
        "dataset_name",
        "epistemic_tag",
        "cosmic_anomaly_index",
        "unknownness_index",
    ],
    title="Frontier Dossier Map — Top Unresolved Objects in Latent Terrain",
)

fig_frontier_dossier_map.add_trace(
    go.Scatter(
        x=frontier_dossier["umap_1"],
        y=frontier_dossier["umap_2"],
        mode="markers+text",
        text=frontier_dossier["frontier_id"],
        textposition="top center",
        marker=dict(
            size=13,
            symbol="x",
            color="red",
            line=dict(width=1, color="black"),
        ),
        name="Top frontier candidates",
    )
)

fig_frontier_dossier_map.update_layout(height=800)
fig_frontier_dossier_map.show()

## Interpretation — Dossier Map

This visualization acts as the investigative layer of the frontier system.

The labelled markers correspond to the highest-ranked unresolved candidates identified by the exploratory pipeline.

Each candidate is repositioned inside the larger latent terrain together with its surrounding representation context.

This allows a human researcher to inspect questions such as:

- Does the candidate emerge inside a coherent anomalous neighbourhood?
- Is it isolated or sector-associated?
- Does it appear near latent-space boundaries or transition regions?
- Is the anomaly potentially linked to weak evidence quality, morphology ambiguity, reconstruction instability, or graph irregularity?

Several patterns become visible immediately:

- many frontier candidates cluster inside specific Euclid sectors,
- some candidates accumulate near sparse manifold boundaries,
- while others emerge inside compact local structures.

The purpose of the dossier map is not to claim discovery.

It is to transform abstract anomaly scores into navigable investigative targets under explicit uncertainty constraints.

This is where speculative visualization becomes operational triage:

a machine-assisted workflow for deciding which unresolved structures deserve deeper inspection.

# Module 8 — Individual Candidate Case Cards

The previous modules analyzed frontier structure at the level of regions, sectors, and latent terrain.

This section shifts back to the scale of individual candidates.

Each frontier object receives a compact visual profile summarizing the balance between multiple unresolvedness dimensions, including:

- anomaly intensity,
- unknownness,
- interpretive risk,
- evidence weakness,
- graph irregularity,
- and reconstruction instability when available.

The purpose of these cards is not classification.

It is comparative inspection.

Different candidates may appear unresolved for very different reasons.

For example:

- one object may exhibit strong anomaly but weak evidence,
- another may appear geometrically isolated,
- another may resist reconstruction,
- while another may accumulate moderate uncertainty across several dimensions simultaneously.

The candidate cards therefore act as exploratory signatures of unresolved representation behaviour.

In this framework, frontier objects are treated less like “discoveries” and more like distinct epistemic profiles inside the broader anomaly landscape.

This creates a more interpretable bridge between:

- large-scale latent cartography,
- and object-level investigative analysis.

In [25]:
# ============================================================
# FRONTIER CASE CARD FUNCTION
# ============================================================

def make_frontier_case_card(row):
    labels = [
        "Unknownness",
        "Anomaly",
        "Interpretive Risk",
        "Low Evidence",
        "Frontier Tension",
    ]

    values = [
        row.get("unknownness_index_norm", np.nan),
        row.get("cosmic_anomaly_index", np.nan),
        row.get("interpretive_risk", np.nan),
        1 - row.get("evidence_score", np.nan),
        row.get("frontier_tension_norm", np.nan),
    ]

    values = [0 if pd.isna(v) else float(v) for v in values]

    fig = go.Figure()

    fig.add_trace(
        go.Scatterpolar(
            r=values + [values[0]],
            theta=labels + [labels[0]],
            fill="toself",
            name=row.get("frontier_id", "frontier candidate"),
        )
    )

    title = (
        f"{row.get('frontier_id', 'Frontier Candidate')} — "
        f"{row.get('dataset_name', 'unknown dataset')} — "
        f"{row.get('epistemic_tag', 'unclassified')}"
    )

    fig.update_layout(
        title=title,
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=500,
        showlegend=False,
    )

    return fig


# Generate cards for top 5 candidates
frontier_case_cards = {}

for _, row in frontier_dossier.head(5).iterrows():
    fig_card = make_frontier_case_card(row)
    fig_name = row["frontier_id"]
    frontier_case_cards[fig_name] = fig_card
    fig_card.show()

## Interpretation — Case Cards

The candidate cards prevent interpretation from collapsing into a single anomaly score.

Each frontier object expresses a different unresolvedness profile across multiple dimensions.

Some candidates appear:

- highly anomalous but relatively evidence-supported,
- weakly evidenced but structurally irregular,
- dominated by interpretive risk,
- or moderately elevated across several categories simultaneously.

This distinction matters because unresolved structure is not necessarily produced by a single mechanism.

Different candidates may emerge from:

- reconstruction instability,
- graph irregularity,
- sparse neighbourhood structure,
- observational weakness,
- latent boundary effects,
- or combinations of several smaller tensions.

The radar profiles therefore function as exploratory signatures of frontier behaviour rather than definitive classifications.

They allow the researcher to compare candidates not only by intensity, but by the shape of their uncertainty structure.

In this sense, the notebook begins to move toward a frontier triage protocol:

a machine-assisted workflow for prioritizing unresolved candidates under explicit interpretive constraints.

# Module 9 — Dataset Frontier Profiles

This section asks a broader systems-level question:

> Which datasets contribute most strongly to the unresolved frontier landscape?

The purpose is not competitive ranking.

It is epistemic profiling.

Different datasets possess different:

- observational properties,
- resolutions,
- feature structures,
- noise characteristics,
- selection effects,
- and geometric behaviours inside latent space.

As a result, different datasets interact with the representation system in different ways.

Some datasets may produce:

- compact and stable latent organization,
- smooth manifold structure,
- and low unresolvedness.

Others may generate:

- fragmented regions,
- sparse topologies,
- elevated anomaly concentration,
- or unstable frontier sectors.

Importantly, this does not imply that one dataset is “better,” “more mysterious,” or closer to new physics.

It means only that the current representation framework experiences different levels of compression difficulty across different observational regimes.

In this sense, the notebook shifts from analyzing individual anomalies toward studying the behaviour of the representation system itself under heterogeneous cosmological data conditions.

In [26]:
dataset_frontier_profile = (
    cosmic_df
    .groupby("dataset_name")
    .agg(
        n=("unknownness_index", "size"),
        mean_unknownness=("unknownness_index", "mean"),
        max_unknownness=("unknownness_index", "max"),
        mean_anomaly=("cosmic_anomaly_index", "mean"),
        mean_evidence=("evidence_score", "mean"),
        mean_risk=("interpretive_risk", "mean"),
    )
    .reset_index()
)

display(dataset_frontier_profile)

fig_dataset_frontier_profile = px.bar(
    dataset_frontier_profile,
    x="dataset_name",
    y=["mean_unknownness", "mean_anomaly", "mean_risk"],
    barmode="group",
    title="Dataset Frontier Profile — Which Representation Regimes Carry the Most Unresolved Structure?",
)

fig_dataset_frontier_profile.update_layout(height=620)
fig_dataset_frontier_profile.show()

,dataset_name,n,mean_unknownness,max_unknownness,mean_anomaly,mean_evidence,mean_risk
0,Euclid,2000,0.250181,0.717452,0.173025,0.634973,0.283754
1,IllustrisTNG,250,0.178302,0.434086,0.107923,0.650000,0.170677
2,SDSS,2000,0.178147,0.487349,0.117229,0.700000,0.186841


## Interpretation — Dataset Frontier Profiles

The dataset frontier profiles summarize how different observational regimes interact with the latent representation system.

Several patterns emerge from the current pipeline configuration.

Euclid contributes the highest average:

- unknownness,
- anomaly intensity,
- and interpretive risk.

At the same time, its evidence score remains relatively strong overall.

This suggests that the Euclid regions are not simply dominated by low-quality detections or obvious noise.

Instead, the representation system appears to experience greater compression tension when organizing Euclid-derived structures inside the shared latent manifold.

By contrast:

- IllustrisTNG appears comparatively stable and lower-risk,
- while SDSS occupies an intermediate regime with broader observational consistency.

Importantly, these results should not be interpreted as evidence of new astrophysical phenomena.

The figure describes the behaviour of the representation system under heterogeneous data conditions.

Several factors may contribute to the observed differences, including:

- feature richness,
- observational depth,
- morphology complexity,
- manifold sparsity,
- domain mismatch,
- or latent geometric fragmentation.

In this sense, the notebook is not only profiling cosmological objects.

It is profiling how machine representations behave across different cosmological observation regimes.

The unresolved frontier therefore emerges partly from the universe being observed, and partly from the limitations and tensions of the representation framework itself.

# Module 10 — Sky Locator for Observational Frontier Candidates

The previous modules analyzed frontier candidates inside latent representation space.

This section reconnects those candidates to observational astronomy coordinates.

For objects originating from observational datasets such as:

- SDSS,
- and Euclid,

the notebook can recover approximate sky positions using:

- Right Ascension (RA),
- and Declination (DEC).

These coordinates allow frontier candidates to be traced back toward their original observational context and source archives.

The purpose is not verification of discoveries.

It is observational grounding.

This creates a bridge between:

- abstract latent-space anomaly structure,
- and physically indexed astronomical observations.

In practical terms, this allows researchers to:

- inspect candidate images directly,
- compare morphology,
- revisit catalogue metadata,
- or cross-reference external archives and surveys.

The module therefore acts as a translation layer between:

- machine-generated frontier cartography,
- and human astronomical investigation.

Only candidates with available sky coordinates are included in this step.

In [27]:
sky_frontier = frontier_dossier.dropna(subset=["ra", "dec"], how="any").copy()

if len(sky_frontier):
    fig_sky_locator = px.scatter(
        sky_frontier,
        x="ra",
        y="dec",
        color="unknownness_index",
        symbol="dataset_name",
        text="frontier_id",
        hover_data=[
            "dataset_name",
            "frontier_id",
            "cosmic_anomaly_index",
            "evidence_score",
            "epistemic_tag",
        ],
        title="Sky Locator — Observational Frontier Candidates",
    )

    fig_sky_locator.update_traces(textposition="top center")
    fig_sky_locator.update_layout(height=650)
    fig_sky_locator.show()

    display(
        sky_frontier[
            [
                "frontier_id",
                "dataset_name",
                "ra",
                "dec",
                "unknownness_index",
                "cosmic_anomaly_index",
                "evidence_score",
                "epistemic_tag",
            ]
        ]
    )
else:
    print("No top frontier candidates with RA/DEC coordinates.")

,frontier_id,dataset_name,ra,dec,unknownness_index,cosmic_anomaly_index,evidence_score,epistemic_tag
2350,CL-FRONTIER-001,Euclid,269.742957,66.033976,0.717452,0.585394,0.479878,speculative
3041,CL-FRONTIER-002,Euclid,269.721246,65.993703,0.697295,0.566341,0.479878,speculative
3487,CL-FRONTIER-003,Euclid,269.782354,66.043101,0.666948,0.547108,0.529878,speculative
3947,CL-FRONTIER-004,Euclid,269.814526,66.008484,0.638287,0.510564,0.479878,speculative
2970,CL-FRONTIER-005,Euclid,269.731482,66.045618,0.636860,0.518667,0.529878,speculative
3698,CL-FRONTIER-006,Euclid,269.649459,66.011309,0.630176,0.534103,0.644945,inferred
3982,CL-FRONTIER-007,Euclid,269.815306,66.008314,0.625660,0.529635,0.643892,inferred
3667,CL-FRONTIER-008,Euclid,269.657764,66.003749,0.620426,0.493680,0.479878,inferred
2741,CL-FRONTIER-009,Euclid,269.728771,65.998099,0.617142,0.490577,0.479878,inferred
3613,CL-FRONTIER-010,Euclid,269.779885,66.045945,0.616340,0.520985,0.644736,inferred


## Interpretation — Sky Locator

The sky locator reconnects frontier candidates to physical observational coordinates.

Unlike the latent manifold views, which describe relationships inside representation space, this figure places candidates back into astronomical sky coordinates using Right Ascension (RA) and Declination (DEC).

Several patterns become immediately visible:

- many frontier candidates occupy nearby coordinate regions,
- some appear in tightly clustered observational neighbourhoods,
- while others remain more spatially isolated.

This does not imply that the objects are physically related.

However, local concentration patterns may indicate:

- shared observational environments,
- survey tiling behaviour,
- morphology clustering,
- feature similarity,
- or common representation-space tensions emerging from nearby observational conditions.

The figure therefore acts as a bridge between:

- machine-generated frontier cartography,
- and traditional observational astronomy workflows.

A researcher could now:

- revisit image cutouts,
- inspect catalogue metadata,
- compare neighbouring candidates,
- or cross-reference external survey archives.

Importantly, the sky locator does not validate the frontier candidates.

It operationalizes them.

The unresolved structures identified earlier in latent space are now transformed into traceable observational targets.

# Closing Interpretation

Notebook 2 extends the governed representation framework of Notebook 1 into an exploratory regime.

The objective is not astrophysical confirmation.

The objective is to operationalize unresolved structure.

The notebook treats latent representation space as an exploratory environment in which:

- anomaly concentration,
- topology fragmentation,
- reconstruction instability,
- sector discontinuity,
- and evidence asymmetry

can be tracked, localized, and organized into investigable frontier candidates.

This produces a computational workflow that behaves less like a traditional anomaly detector and more like a speculative survey instrument operating over representation geometry.

The resulting frontier objects are not interpreted as discoveries.

They are treated as unresolved compression events inside a heterogeneous cosmological representation system.

Some candidates may ultimately correspond to:

- observational artifacts,
- morphology edge cases,
- sparse manifold regions,
- cross-domain representation mismatch,
- or ordinary astrophysical structure amplified by the latent geometry.

However, the notebook deliberately preserves these candidates rather than collapsing them immediately into dismissal or confirmation.

This creates a controlled exploratory layer positioned between:

- statistical anomaly detection,
- observational astronomy,
- and speculative hypothesis generation.

The central methodological constraint is that speculation remains attached to explicit operational variables:

- evidence quality,
- interpretive risk,
- latent position,
- topological behaviour,
- and reconstruction dynamics.

In this framework, speculative science is treated not as unrestricted interpretation, but as structured exploration under computational and epistemic constraints.

The notebook therefore functions as an experimental cartographic system for identifying and organizing unresolved structure in large-scale cosmological representation spaces.

In [28]:
# ============================================================
# EXPORT FRONTIER NOTEBOOK OUTPUTS
# ============================================================

import os
import zipfile

FRONTIER_EXPORT_DIR = "/content/cosmic_latent_frontier_exports"
os.makedirs(FRONTIER_EXPORT_DIR, exist_ok=True)

# Tables
cosmic_df.to_csv(
    os.path.join(FRONTIER_EXPORT_DIR, "cosmic_latent_frontier_enriched.csv"),
    index=False
)

frontier_dossier.to_csv(
    os.path.join(FRONTIER_EXPORT_DIR, "cosmic_latent_frontier_dossier.csv"),
    index=False
)

sector_summary.to_csv(
    os.path.join(FRONTIER_EXPORT_DIR, "cosmic_latent_frontier_sector_summary.csv"),
    index=False
)

dataset_frontier_profile.to_csv(
    os.path.join(FRONTIER_EXPORT_DIR, "cosmic_latent_dataset_frontier_profile.csv"),
    index=False
)

# Figures
frontier_figures = {
    "fig_epistemic_frontier": globals().get("fig_epistemic_frontier"),
    "fig_frontier_quadrants": globals().get("fig_frontier_quadrants"),
    "fig_frontier_sectors": globals().get("fig_frontier_sectors"),
    "fig_frontier_dossier_map": globals().get("fig_frontier_dossier_map"),
    "fig_anomaly_constellation": globals().get("fig_anomaly_constellation"),
    "fig_dataset_frontier_profile": globals().get("fig_dataset_frontier_profile"),
    "fig_sky_locator": globals().get("fig_sky_locator"),
}

for name, fig in frontier_figures.items():
    if isinstance(fig, go.Figure):
        fig.write_html(os.path.join(FRONTIER_EXPORT_DIR, f"{name}.html"))
        try:
            fig.write_image(os.path.join(FRONTIER_EXPORT_DIR, f"{name}.png"), scale=2)
        except Exception as e:
            print(f"PNG export failed for {name}: {repr(e)}")
            print("Install kaleido if PNG export is needed: !pip install -U kaleido")

# Case cards
for card_id, fig in frontier_case_cards.items():
    safe_id = card_id.replace("/", "_")
    fig.write_html(os.path.join(FRONTIER_EXPORT_DIR, f"case_card_{safe_id}.html"))

# Metadata
metadata = {
    "project": "COSMIC-LATENT FRONTIER",
    "notebook": "Notebook 2 — Speculative Cartography Annex",
    "n_rows": int(len(cosmic_df)),
    "n_frontier_candidates": int(len(frontier_dossier)),
    "datasets": list(cosmic_df["dataset_name"].dropna().unique()),
}

with open(os.path.join(FRONTIER_EXPORT_DIR, "frontier_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

zip_path = "/content/cosmic_latent_frontier_exports.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(FRONTIER_EXPORT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, FRONTIER_EXPORT_DIR)
            zipf.write(full_path, arcname)

print("Frontier exports saved to:", FRONTIER_EXPORT_DIR)
print("ZIP archive:", zip_path)

PNG export failed for fig_epistemic_frontier: ValueError('\nImage export using the "kaleido" engine requires the kaleido package,\nwhich can be installed using pip:\n    $ pip install -U kaleido\n')
Install kaleido if PNG export is needed: !pip install -U kaleido
PNG export failed for fig_frontier_quadrants: ValueError('\nImage export using the "kaleido" engine requires the kaleido package,\nwhich can be installed using pip:\n    $ pip install -U kaleido\n')
Install kaleido if PNG export is needed: !pip install -U kaleido
PNG export failed for fig_frontier_sectors: ValueError('\nImage export using the "kaleido" engine requires the kaleido package,\nwhich can be installed using pip:\n    $ pip install -U kaleido\n')
Install kaleido if PNG export is needed: !pip install -U kaleido
PNG export failed for fig_frontier_dossier_map: ValueError('\nImage export using the "kaleido" engine requires the kaleido package,\nwhich can be installed using pip:\n    $ pip install -U kaleido\n')
Install k